In [13]:
import pandas as pd
import torch
import numpy as np
from rdkit import Chem
import json
from IPython.display import display

In [4]:
def is_valid_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        return mol is not None
    except:
        return False

In [5]:
MAX_LEN = 128
BATCH_SIZE = 128
EMB_DIM = 256
LATENT_DIM = 128
N_HEADS = 8
FF_DIM = 512
NUM_LAYERS = 4
VOCAB_SPECIAL = ['<pad>', '<bos>', '<eos>', '<unk>']

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используем устройство: {DEVICE}")

Используем устройство: cuda


In [7]:
# Загружаем датасет
df = pd.read_csv("data/polymers_with_names_predicted.csv")

with open("data/user_request_dataset/property_mapping.json", "r", encoding="utf-8") as f:
    json_data = json.load(f)
    PROPERTY_MAPPING = json_data["PROPERTY_MAPPING"] 

# признаки
NUM_FEATURES = list(PROPERTY_MAPPING.keys())
print(f"Количество признаков: {len(NUM_FEATURES)}")

Количество признаков: 37


In [11]:
class PolymerSelector:
    def __init__(self, dataset_path):

        self.df = pd.read_csv(dataset_path)
        print(f"База данных загружена: {len(self.df)} полимеров.")
        
        self.column_mapping = {
            'Name': ['name', 'polymer_name', 'Polymer Name', 'Name'],
            'Polymer_SMILES': ['smiles', 'poly_smiles', 'canonical_smiles', 'Polymer_SMILES'],
            'Monomer_SMILES_1': ['monomer_1', 'smiles_mono_1', 'Monomer_SMILES_1', 'Monomer 1'],
            'Monomer_SMILES_2': ['monomer_2', 'smiles_mono_2', 'Monomer_SMILES_2', 'Monomer 2']
        }
        
        # Находим реальные имена колонок в загруженном файле
        self.final_cols = {}
        for target_name, alternatives in self.column_mapping.items():
            for alt in alternatives:
                # Ищем точное совпадение
                if alt in self.df.columns:
                    self.final_cols[target_name] = alt
                    break
                # Или ищем, если название колонки содержит искомую подстроку (например "smiles" внутри "my_smiles_col")
                # (но аккуратно, чтобы smiles не спутать с smiles_1)
                matches = [c for c in self.df.columns if alt.lower() == c.lower()]
                if matches:
                    self.final_cols[target_name] = matches[0]
                    break
        
    def find_best_matches(self, json_data, top_n=5):

        # 1. Загрузка JSON
        if isinstance(json_data, str):
            with open(json_data, 'r', encoding='utf-8') as f:
                reqs = json.load(f)
        else:
            reqs = json_data

        # 2. Создаем копию для расчетов
        candidates = self.df.copy()
        
        # Начинаем с нулевого штрафа
        candidates['similarity_score'] = 0.0
        used_props = []

        print("\n--- Расчет отклонений ---")
        for prop, details in reqs.items():
            # Пропускаем, если свойства нет в CSV
            if prop not in candidates.columns:
                print(f"ПРОПУСК: Свойство '{prop}' отсутствует в базе данных.")
                continue
            
            # Получаем целевое значение
            if isinstance(details, dict) and 'target_value' in details:
                target = float(details['target_value'])
            else:
                # На случай если json старого формата просто "key": value
                target = float(details)
            
            epsilon = 1e-9 # Защита от деления на 0
            
            # Векторизованное вычисление для всей колонки сразу
            diff = np.abs(candidates[prop] - target)
            normalized_error = diff / (np.abs(target) + epsilon)
            
            # Добавляем к общему счету
            candidates['similarity_score'] += normalized_error
            used_props.append(prop)

        if not used_props:
            print("Не удалось найти ни одного совпадающего свойства.")
            return pd.DataFrame()

        # 4. Сортировка (чем меньше score, тем ближе полимер)
        # score = 0 означает идеальное совпадение
        best_matches = candidates.sort_values(by='similarity_score', ascending=True).head(top_n)

        # 5. Формирование красивого вывода
        output_columns = []
        
        # Добавляем идентификаторы (Имя, Смайлзы) с переименованием
        rename_dict = {}
        for target_name, csv_name in self.final_cols.items():
            output_columns.append(target_name)
            rename_dict[csv_name] = target_name
            
        # Добавляем колонки со свойствами, чтобы видеть значения
        output_columns.extend(used_props)
        output_columns.append('similarity_score') # Чтобы видеть, насколько хорош матч
        
        # Переименовываем и выбираем колонки
        result = best_matches.rename(columns=rename_dict)
        
        # Если какой-то колонки (например Monomer 2) не было в файле, создаем её пустой для красоты
        for col in ['Monomer_SMILES_2']: 
            if col not in result.columns:
                result[col] = "" # Пустая строка
        
        # Убедимся, что выбираем только существующие колонки
        existing_cols = [c for c in output_columns if c in result.columns]
        result = result[existing_cols]
        
        # Заполняем NaN в мономерах пустыми строками
        if 'Monomer_SMILES_2' in result.columns:
            result['Monomer_SMILES_2'] = result['Monomer_SMILES_2'].fillna("")
            
        return result

In [14]:
nlp_json = 'data/nlp_to_gen_bridge.json'
with open(nlp_json, 'r', encoding='utf-8') as f:
            json_params = json.load(f)

pd.set_option('display.max_columns', None)      # Показать все колонки
pd.set_option('display.max_colwidth', None)     # Не обрезать длинные строки (SMILES)
pd.set_option('display.precision', 4)           # 4 знака после запятой для чисел

# 2. Инициализация
# Убедитесь, что файл существует (создан на прошлом шаге)
selector = PolymerSelector("data/polymers_with_names_predicted.csv")

# 3. Поиск
top_polymers = selector.find_best_matches(nlp_json, top_n=5)

# 4. Вывод
print("\n=== Топ 5 подходящих полимеров ===")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Выводим без индекса для красоты
display(top_polymers)

База данных загружена: 11239 полимеров.
Найденные колонки: {'Name': 'Name', 'Polymer_SMILES': 'Polymer_SMILES', 'Monomer_SMILES_1': 'Monomer_SMILES_1', 'Monomer_SMILES_2': 'Monomer_SMILES_2'}

--- Расчет отклонений ---
ПРОПУСК: Свойство 'Eat' отсутствует в базе данных.

=== Топ 5 подходящих полимеров ===


,Name,Polymer_SMILES,Monomer_SMILES_1,Monomer_SMILES_2,LOI,epsc,permHe,Egc,rho,Egb,Eib,CED,Ei,Eea,nc,ne,Xc,Xe,epse_6.0,epse_3.0,epse_1.78,epse_15.0,epse_4.0,epse_5.0,epse_2.0,epse_9.0,epse_7.0,epsb,TSb,TSy,YM,permCH4,permCO2,permH2,permO2,permN2,Cp,Td,Tg,Tm,similarity_score
5609,"poly[(3,3'-disulfanylbenzidine)-alt-(diphenyl 4,4'-oxydibenzoate)]",*c1nc2ccc(cc2s1)-c1ccc2nc(sc2c1)-c1ccc(Oc2ccc(*)cc2)cc1,Nc1ccc(cc1S)-c1ccc(N)c(S)c1,O=C(Oc1ccccc1)c1ccc(Oc2ccc(cc2)C(=O)Oc2ccccc2)cc1,30.9073,4.4821,15.1638,3.2507,1.2933,2.8121,3.3681,125.5406,5.5877,2.2517,2.0036,1.6615,37.8863,34.1172,3.3350,3.5974,3.9182,2.6799,3.4753,3.3991,3.8404,3.1096,3.2651,11.1049,85.9046,63.4949,2684.7783,0.6042,8.7513,16.3665,2.2456,0.5714,1.1708,713.9310,492.3359,612.1606,4.7761
10422,poly[9-(oxiran-2-ylmethyl)-9H-carbazole],*c1ccc2n(CC3CO3)c3ccc(*)cc3c2c1,C(C1CO1)n1c2ccccc2c2ccccc12,,25.2793,4.5145,15.5117,3.9603,1.2712,3.4401,3.7315,131.9619,5.7649,1.9380,2.0131,1.6616,24.7081,22.4131,3.2878,3.5429,3.9055,2.7115,3.4212,3.3443,3.8214,3.0910,3.2249,10.5634,76.3500,69.3441,2038.6080,1.1437,9.0173,16.2325,2.8346,0.8596,1.0467,715.9122,515.1489,630.3774,4.8123
2214,"poly[(1,1'-biphenyl-4,4'-dithiol)-alt-(isophthaloyl dichloride)]",*SC(=O)c1cccc(c1)C(=O)Sc1ccc(cc1)-c1ccc(*)cc1,Sc1ccc(cc1)-c1ccc(S)cc1,ClC(=O)c1cccc(c1)C(Cl)=O,31.0377,4.3004,14.8877,3.7157,1.2968,3.1404,3.4151,137.1871,6.1282,2.4912,1.9313,1.6119,44.3511,38.2648,3.4570,3.7830,4.0177,2.6233,3.6630,3.5610,3.9657,3.1339,3.3509,11.6937,70.5746,56.9993,2064.5942,0.8688,10.4431,17.1966,2.6328,0.7884,1.2122,624.2748,408.4961,523.8438,4.9233
6047,"poly{[2,4-bis(p-fluorophenyl)-6-phenyl-s-triazine]-alt-(naphthalene-2,7-diol)}",*c1ccc(Oc2ccc3ccc(Oc4ccc(cc4)-c4nc(*)nc(n4)-c4ccccc4)cc3c2)cc1,Fc1ccc(cc1)-c1nc(nc(n1)-c1ccc(F)cc1)-c1ccccc1,Oc1ccc2ccc(O)cc2c1,31.6153,4.3661,16.6301,3.4591,1.3000,3.1938,3.4321,118.5143,5.6101,1.9853,1.9484,1.6330,42.3428,38.2874,3.2198,3.4791,3.7603,2.5916,3.3654,3.2883,3.6919,2.9862,3.1474,9.8492,83.7006,70.9517,2154.2468,0.6339,6.5584,16.7533,1.9307,0.6001,1.2625,748.9647,497.4996,594.4082,4.9889
7033,"poly[(4,4'-sulfanediyldibenzaldehyde)-alt-(biphenyl-4,4'-diamine)]",*Sc1ccc(C=Nc2ccc(cc2)-c2ccc(cc2)N=Cc2ccc(*)cc2)cc1,O=Cc1ccc(Sc2ccc(C=O)cc2)cc1,Nc1ccc(cc1)-c1ccc(N)cc1,31.2291,4.5688,14.8666,3.2720,1.2708,2.8682,3.4423,125.0373,5.4839,2.1731,2.0102,1.6562,39.8229,36.8128,3.5271,3.8444,4.1507,2.7241,3.7097,3.6144,4.0773,3.2392,3.4337,11.1415,82.1192,72.6234,2249.3906,0.7152,7.8657,15.3566,2.2212,0.6437,1.2455,647.1782,417.5021,535.3580,5.0408
